# 一格一格看懂 `tokenize_example`

这个笔记本只加载 **Qwen 分词器**，不加载 0.5B 模型、不训练，也不需要 GPU。按顺序运行每个代码格。这里的变量名和 `manual_train.py` 中的函数保持一致。

## 0. 下载来源与示例数据

默认使用 HF 镜像。若你已从魔搭下载模型，可以在启动 Jupyter 前设置环境变量 `STARTER_MODEL` 为本地模型目录；本格不会覆盖它。`HF_ENDPOINT` 要在导入 `transformers` 前设置。

In [1]:
import os

# 仅在没有设置 HF_ENDPOINT 时使用镜像；已经下载到本地则不会访问它。
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
model_id = os.environ.get("STARTER_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
example = {"prompt": "什么是 LoRA？", "completion": "LoRA 是低秩微调。"}
max_length = 64
print("模型位置:", model_id)
print("样本:", example)
print("长度上限:", max_length)

模型位置: Qwen/Qwen2.5-0.5B-Instruct
样本: {'prompt': '什么是 LoRA？', 'completion': 'LoRA 是低秩微调。'}
长度上限: 64


## 1. 加载分词器

`AutoTokenizer` 读取词表和聊天模板。`pad_token` 设为结束符，与训练脚本一致。这里下载的是分词器文件，不是模型权重。

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
print("分词器类型:", type(tokenizer).__name__)
print("结束符:", repr(tokenizer.eos_token), tokenizer.eos_token_id)
print("填充符:", repr(tokenizer.pad_token), tokenizer.pad_token_id)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

d:\app\miniconda3\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shaolei\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

分词器类型: Qwen2Tokenizer
结束符: '<|im_end|>' 151645
填充符: '<|im_end|>' 151645


## 2. `apply_chat_template`：普通问题 → 带角色的字符串

观察 `repr(prompt_text)` 中的 `\n`。`tokenize=False` 表示暂时只生成字符串；`add_generation_prompt=True` 在末尾加入 assistant 开头。

In [11]:
example = {"prompt": "对不起", "completion": "没关系"}
max_length = 64
prompt_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": example["prompt"]}],
    tokenize=False,
    add_generation_prompt=True,
)
print("普通打印:")
print(prompt_text)
# repr() 会返回一个对象的便于检查的字符串表示。它特别适合看见普通 print() 不容易发现的字符，比如换行 \n、空格和引号。
print("repr 打印（能看见换行符）:", repr(prompt_text))

普通打印:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
对不起<|im_end|>
<|im_start|>assistant

repr 打印（能看见换行符）: '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n对不起<|im_end|>\n<|im_start|>assistant\n'


In [13]:
text = "你好"
a = tokenizer(text, add_special_tokens=False)
b = tokenizer(text, add_special_tokens=True)
print(f"不加特殊符号: {a['input_ids']}, attention_mask: {a['attention_mask']}")
print(f"加特殊符号: {b['input_ids']}, attention_mask: {b['attention_mask']}")

不加特殊符号: [108386], attention_mask: [1]
加特殊符号: [108386], attention_mask: [1]


## 3. `prompt_ids`：字符串 → token ID 列表

此时才进行分词。聊天模板已经放入角色标记，所以 `add_special_tokens=False` 不再额外添加标记。`len(prompt_ids)` 决定后面多少个标签要被忽略。

In [14]:
# return a dict with "input_ids" and "attention_mask"
print(prompt_text)
prompt = tokenizer(prompt_text, add_special_tokens=False)
print(f"input_ids: {prompt['input_ids']}")
print(f"attention_mask: {prompt['attention_mask']}")

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
对不起<|im_end|>
<|im_start|>assistant

input_ids: [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 109024, 151645, 198, 151644, 77091, 198]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [9]:
print(prompt_text)
prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
print("after tokenizer:")
print(prompt_ids)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
什么是 LoRA？<|im_end|>
<|im_start|>assistant

after tokenizer:
[151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 106582, 6485, 5609, 11319, 151645, 198, 151644, 77091, 198]


## 4. `full_text`：接上目标回答和结束符

这里的 `+` 是字符串拼接。完整文本由「带角色的提问」「标准回答」「结束符」三段组成。

In [ ]:
full_text = prompt_text + example["completion"] + tokenizer.eos_token
print(full_text)
print("完整文本长度（字符）:", len(full_text))
print("完整文本长度（token）:", len(tokenizer(full_text, add_special_tokens=False)["input_ids"]))

## 5. `encoded`：完整文本 → 定长输入

`truncation=True` 截掉超过 64 个 token 的末尾；`padding="max_length"` 把不足 64 的部分补齐。`attention_mask` 的 1 是真实内容，0 是填充。若下面真实 token 数已达到 64，可把第一格的 `max_length` 调大后重跑。

In [ ]:
encoded = tokenizer(
    full_text,
    add_special_tokens=False,
    truncation=True,
    max_length=max_length,
    padding="max_length",
)
print("返回类型:", type(encoded))
print("字段:", list(encoded.keys()))
print("总长度:", len(encoded["input_ids"]))
print("真实 token 数:", sum(encoded["attention_mask"]))
print("填充 token 数:", encoded["attention_mask"].count(0))
print("input_ids 前 20 个:", encoded["input_ids"][:20])
print("attention_mask 后 20 个:", encoded["attention_mask"][-20:])

## 6. `labels`：先复制 `input_ids`

`.copy()` 创建独立列表。接下来修改 `labels` 时，`encoded["input_ids"]` 不会一起改变。模型内部会把标签错开一位，计算预测下一个 token 的损失。

In [ ]:
labels = encoded["input_ids"].copy()
print("初始 labels 等于 input_ids 吗？", labels == encoded["input_ids"])
print("两个列表是同一对象吗？", labels is encoded["input_ids"])

## 7. 忽略提问部分

`-100` 是交叉熵损失的忽略值。把前 `len(prompt_ids)` 个位置改成 `-100`，让模型只对回答部分学习。

In [ ]:
prompt_length = min(len(prompt_ids), len(labels))
labels[:prompt_length] = [-100] * prompt_length
print("被忽略的 prompt 位置数:", prompt_length)
print("labels 前 25 个:", labels[:25])
print("input_ids 前 25 个（仍是原值）:", encoded["input_ids"][:25])

## 8. 忽略填充部分

列表推导式逐一检查 `attention_mask`：为 1 就保留当前标签，为 0 就改为 `-100`。

In [ ]:
labels = [label if mask else -100 for label, mask in zip(labels, encoded["attention_mask"])]
print("labels 后 20 个:", labels[-20:])
print("参与 loss 的 token 数:", sum(label != -100 for label in labels))
print("填充位置都被忽略了吗？", all(label == -100 for label, mask in zip(labels, encoded["attention_mask"]) if mask == 0))

## 9. 逐位置查看最终结果

一行代表一个 token 位置。`训练目标` 是 `回答` 才参与 loss；提问和补齐都被忽略。特殊标记的显示由 Qwen 词表决定。

In [ ]:
for position, (token_id, mask, label) in enumerate(zip(encoded["input_ids"], encoded["attention_mask"], labels)):
    region = "填充" if mask == 0 else ("提问" if position < len(prompt_ids) else "回答")
    token = tokenizer.convert_ids_to_tokens(token_id)
    print(f"{position:02d}  {region:2s}  id={token_id:6d}  mask={mask}  label={label:6d}  token={token!r}")

## 10. 对照函数的返回值

这就是 `tokenize_example` 返回的字典。到这一步它仍包含 Python 列表；训练脚本稍后调用 `dataset.set_format(type="torch")`，取样时才得到张量。

In [ ]:
if all(label == -100 for label in labels):
    raise ValueError("目标回答被截断；请增大 max_length")
result = {**encoded, "labels": labels}
print("返回字段:", list(result.keys()))
print("各字段长度:", {key: len(value) for key, value in result.items()})

## 自己动手改三个地方

1. 把 `example["prompt"]` 和 `example["completion"]` 改成自己的句子，再从第 2 格重跑。
2. 把 `max_length` 改大，观察填充位置怎样增加。
3. 临时把 `max_length` 改为小于 `len(prompt_ids)` 的值，观察为什么训练样本会报“回答被截断”。改完后恢复 64。